# Pipeline 1 - Amostragem natural de 40.000 artigos

Le o snapshot completo do arXiv em streaming e usa reservoir sampling para obter
40.000 artigos sem favorecer os primeiros registros. Somente entram artigos cuja
categoria primaria pertence as 15 classes do projeto. Todos os rotulos-alvo do
campo `categories` sao preservados para o experimento multi-label.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import os
import random
from collections import Counter
from pathlib import Path

SEED = 42
N_AMOSTRA = 40000
NOME_SAIDA = 'arxiv_amostra_40000_multilabel.json'

TARGET_CATS = [
    'cs.LG', 'cs.AI', 'cs.CL', 'cs.CV', 'hep-ph', 'hep-th', 'gr-qc',
    'quant-ph', 'astro-ph', 'math-ph', 'math.MP', 'cond-mat.mtrl-sci',
    'cond-mat.mes-hall', 'cond-mat.str-el', 'cond-mat.stat-mech'
]


## Localizacao dos arquivos

A busca e limitada a poucos niveis para evitar varrer o Drive inteiro. Ajuste
`BASE` se o atalho do projeto tiver outro nome.


In [ ]:
NOME_SNAPSHOT = 'arxiv-metadata-oai-snapshot.json'
MY_DRIVE = Path('/content/drive/MyDrive')
SHARED_DRIVES = Path('/content/drive/Shareddrives')

# Busca rasa: raiz conhecida, pastas diretamente em MyDrive e Shared Drives.
# Evita glob recursivo, que pode levar varios minutos no Google Drive.
pastas_projeto = []
if MY_DRIVE.exists():
    pastas_projeto.append(MY_DRIVE)
    primeiro_nivel = [p for p in MY_DRIVE.iterdir() if p.is_dir()]
    pastas_projeto.extend(primeiro_nivel)
    for pasta in primeiro_nivel:
        try:
            pastas_projeto.extend(p for p in pasta.iterdir() if p.is_dir())
        except (OSError, PermissionError):
            pass

if SHARED_DRIVES.exists():
    for drive_compartilhado in SHARED_DRIVES.iterdir():
        if drive_compartilhado.is_dir():
            pastas_projeto.append(drive_compartilhado)
            primeiro_nivel = [p for p in drive_compartilhado.iterdir() if p.is_dir()]
            pastas_projeto.extend(primeiro_nivel)
            for pasta in primeiro_nivel:
                try:
                    pastas_projeto.extend(p for p in pasta.iterdir() if p.is_dir())
                except (OSError, PermissionError):
                    pass

candidatos = []
for pasta in pastas_projeto:
    candidatos.extend([
        pasta / 'archive' / NOME_SNAPSHOT,
        pasta / NOME_SNAPSHOT,
    ])

ARQUIVO_FONTE = next((p for p in candidatos if p.is_file()), None)
if ARQUIVO_FONTE is None:
    pastas_visiveis = ', '.join(str(p) for p in pastas_projeto[:80])
    raise FileNotFoundError(
        'Snapshot nao encontrado em uma busca rasa. Pastas visiveis: '
        + pastas_visiveis
        + '. Ajuste ARQUIVO_FONTE usando o caminho mostrado no painel Arquivos do Colab.'
    )

# Localiza a pasta pipelines ja usada pelos experimentos anteriores.
# No Drive atual, o snapshot fica em projeto/source-arxiv/archive, enquanto
# as saidas ficam em projeto/pipelines.
raiz_fonte = ARQUIVO_FONTE.parent.parent if ARQUIVO_FONTE.parent.name == 'archive' else ARQUIVO_FONTE.parent
candidatos_base = [raiz_fonte, *list(raiz_fonte.parents)]

def tem_pipeline_anterior(base):
    pasta = base / 'pipelines'
    return pasta.is_dir() and (
        any(pasta.glob('arxiv_amostra_10500*'))
        or any(pasta.glob('pipeline*_colab*.ipynb'))
        or any(pasta.glob('pipeline*10500*.ipynb'))
    )

BASE = next((base for base in candidatos_base if tem_pipeline_anterior(base)), raiz_fonte)
PIPE = BASE / 'pipelines'
PIPE.mkdir(parents=True, exist_ok=True)
SAIDA = PIPE / NOME_SAIDA

print('Entrada:', ARQUIVO_FONTE)
print('Base de saida:', BASE)
print('Saida:', SAIDA)


## Reservoir sampling reproduzivel


In [ ]:
rng = random.Random(SEED)
reservatorio = []
elegiveis = 0
linhas_lidas = 0

with ARQUIVO_FONTE.open('r', encoding='utf-8') as f:
    for linha in f:
        linhas_lidas += 1
        try:
            paper = json.loads(linha)
        except json.JSONDecodeError:
            continue

        cats = (paper.get('categories') or '').split()
        if not cats or cats[0] not in TARGET_CATS:
            continue

        labels = [cat for cat in cats if cat in TARGET_CATS]
        if not labels:
            continue

        registro = {
            'id': paper.get('id'),
            'title': paper.get('title'),
            'abstract': paper.get('abstract'),
            'categories': paper.get('categories'),
            'primary_category': cats[0],
            'target_categories': labels,
            'authors': paper.get('authors'),
            'update_date': paper.get('update_date'),
        }

        elegiveis += 1
        if len(reservatorio) < N_AMOSTRA:
            reservatorio.append(registro)
        else:
            j = rng.randrange(elegiveis)
            if j < N_AMOSTRA:
                reservatorio[j] = registro

print('Linhas lidas:', linhas_lidas)
print('Artigos elegiveis:', elegiveis)
print('Amostra:', len(reservatorio))
if len(reservatorio) != N_AMOSTRA:
    raise RuntimeError(f'Esperados {N_AMOSTRA}, encontrados {len(reservatorio)}')


## Validacao e gravacao


In [ ]:
primary_counts = Counter(p['primary_category'] for p in reservatorio)
label_counts = Counter(label for p in reservatorio for label in p['target_categories'])
cardinalidade = sum(len(p['target_categories']) for p in reservatorio) / len(reservatorio)

print('Distribuicao primaria:')
print(dict(primary_counts.most_common()))
print()
print('Distribuicao multi-label:')
print(dict(label_counts.most_common()))
print()
print('Cardinalidade media de rotulos:', round(cardinalidade, 3))

with SAIDA.open('w', encoding='utf-8') as f:
    for registro in reservatorio:
        f.write(json.dumps(registro, ensure_ascii=False) + chr(10))

print('Arquivo salvo:', SAIDA, SAIDA.stat().st_size, 'bytes')
